
# Baseline de clasificacion de urgencia: TF-IDF + Logistic Regression.
Fuente de datos: subset en espanol del dataset de Kaggle + correos reales
propios ya etiquetados (si existen).


#### Bibliotecas

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

load_dotenv()

python-dotenv could not parse statement starting at line 31


True

### 1. Cargar los datos del dataset de Kaggle ( español )

In [16]:
print(os.getcwd())

/home/jorge/proyectos/sicpc-thesis


In [20]:
#os.chdir("sicpc-thesis")  # sube un nivel, a la raiz del proyecto
df_kaggle = pd.read_csv("data/processed/kaggle_es_baseline.csv")
#df_kaggle = pd.read_csv("data/raw/kaggle/tickets.csv")
df_kaggle["texto"] = df_kaggle["subject"].fillna("") + " " + df_kaggle["body"].fillna("")
df_kaggle = df_kaggle[["texto", "urgencia"]]
df_kaggle["fuente"] = "kaggle"
#print(df_kaggle.head())
print(f"Filas de Kaggle (ES): {len(df_kaggle)}")

Filas de Kaggle (ES): 812


In [21]:
print(df_kaggle.columns.tolist())

['texto', 'urgencia', 'fuente']


###  2. Cargar correos reales ya etiquetados (si hay)

In [23]:
df_reales = pd.DataFrame(columns=["texto", "urgencia", "fuente"])

try:
    import mysql.connector

    conn = mysql.connector.connect(
        host="localhost", port=3306,
        user="sJorge", password=os.getenv("MYSQL_PASSWORD"),
        database="sicpc",
    )
    df_reales = pd.read_sql(
        "SELECT asunto_anonimizado, cuerpo_anonimizado, urgencia_real "
        "FROM correo WHERE urgencia_real IS NOT NULL",
        conn,
    )
    conn.close()

    if len(df_reales) > 0:
        df_reales["texto"] = (
            df_reales["asunto_anonimizado"].fillna("") + " " +
            df_reales["cuerpo_anonimizado"].fillna("")
        )
        df_reales = df_reales.rename(columns={"urgencia_real": "urgencia"})
        df_reales = df_reales[["texto", "urgencia"]]
        df_reales["fuente"] = "real"

    print(f"Filas reales etiquetadas: {len(df_reales)}")
except Exception as e:
    print(f"No se pudieron cargar correos reales (se continua solo con Kaggle): {e}")

No se pudieron cargar correos reales (se continua solo con Kaggle): 1045 (28000): Access denied for user 'sJorge'@'172.19.0.1' (using password: NO)


### 3. Combinar

In [ ]:
df = pd.concat([df_kaggle, df_reales], ignore_index=True)
df = df.dropna(subset=["texto", "urgencia"])
df = df[df["texto"].str.strip() != ""]

print(f"\nTotal combinado: {len(df)} filas")
print(df["urgencia"].value_counts())
print(df["fuente"].value_counts())

### 4. Split train/test (estratificado por urgencia)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"], df["urgencia"],
    test_size=0.2, random_state=42, stratify=df["urgencia"],
)

### 5. Vectorizacion TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### 6. Entrenamiento

In [ ]:
modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo.fit(X_train_tfidf, y_train)

### 7. Evaluacion

y_pred = modelo.predict(X_test_tfidf)

print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACION")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("MATRIZ DE CONFUSION")
print(confusion_matrix(y_test, y_pred, labels=["alta", "media", "baja"]))
print("(orden de filas/columnas: alta, media, baja)")

### 8. Guardar el modelo y el vectorizador para reutilizar despues

In [ ]:
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(modelo, "models/baseline_logistic_regression.pkl")
joblib.dump(vectorizer, "models/baseline_tfidf_vectorizer.pkl")
print("\nModelo y vectorizador guardados en models/")